In [1]:
import os
from typing import List, Dict, Any
import pandas as pd
import langchain

In [21]:
# loading 2 pdfs for RPA
import os
from langchain_community.document_loaders import PyMuPDFLoader

pdf_paths = ["data/attention.pdf","data/python.pdf","data/BERT.pdf","data/Pytorch.pdf"]
documents = []

for path in pdf_paths:
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    
    paper_name = os.path.basename(path)

    for doc in docs:
        doc.metadata["paper"] = paper_name
        doc.metadata["page"] = doc.metadata.get("page",0)
    
    documents.extend(docs)

print(f"Total Pages Loaded: {len(documents)}")

Total Pages Loaded: 51


In [22]:
# chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 444


In [ ]:
# embedding using hugging face embeddings

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6358.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
embedding_model

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [25]:
# store in vecctor database (FAISS)
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(chunks,embedding_model)

vector_db.save_local("faiss_index")

In [26]:
#checking embeddings for our chunks
texts = [doc.page_content for doc in chunks]

embeddings = embedding_model.embed_documents(texts)

print(len(embeddings))        # number of chunks
print(len(embeddings[0]))     # dimension
print(embeddings[0][:100])     # sample values

444
384
[0.11324582248926163, -0.008517144247889519, 0.00959861557930708, 0.05026848614215851, 0.007993432693183422, -0.002763142576441169, -0.03670583665370941, -0.0010259143309667706, -0.028932709246873856, 0.14072135090827942, 0.07664524763822556, -0.04910200089216232, 0.002275572158396244, 0.028174027800559998, -0.06316042691469193, 0.012929757125675678, -0.011600383557379246, -0.010148313827812672, -0.04147076606750488, 0.0763549953699112, -0.02833164855837822, -0.012957349419593811, 0.14418160915374756, -0.07282605022192001, 0.1029583141207695, -0.029230710119009018, -0.03056846372783184, -0.050917115062475204, 0.011760346591472626, 0.01574762351810932, -0.03239961713552475, 0.05222383514046669, 0.00638082018122077, 0.05508680269122124, -0.012266440317034721, -0.007810387760400772, 0.0641307607293129, 0.03087286278605461, 0.03909524902701378, 0.0011153594823554158, 0.0004612210614141077, -0.09159884601831436, 0.022328468039631844, -0.010336479172110558, 0.014856324531137943, 0.05

In [52]:
# creating retriever
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "lambda_mult": 0.7
    }
)

In [53]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020E57471CD0>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.7})

In [54]:
# connecting llm
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    temperature=0,
    model = "openai/gpt-oss-120b"
)

In [55]:
#qa chain
query = "What is transformer and how Python is used in AI?"

In [56]:
vectorsotre = FAISS.from_documents(docs, embedding_model)

In [88]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer ONLY from the provided context.
                                          
Also include citations in this format:
(Source: paper_name, Page number)
                                                  
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{input}

Answer:
""")
#document_chain = create_stuff_documents_chain(llm, prompt)

#qa_chain = create_retrieval_chain(retriever, document_chain)

In [101]:
query = "What is Embeddings and Softmax?"

response = qa_chain.invoke({"input": query})

answer = response["answer"]
sources = response.get("context", [])

In [102]:
print("\n==============================")
print("Q&A Result")
print("==============================")

print(f"Q: {query}\n")

print("Answer:")
print(answer)

print("\nSources:")

seen = set()
for doc in sources:
    paper = doc.metadata.get("paper", "Unknown")
    page = doc.metadata.get("page", "N/A")
    
    key = (paper, page)
    if key not in seen:
        seen.add(key)
        print(f"Source: {paper}, Page {page}")
        break

print("==============================\n")


Q&A Result
Q: What is Embeddings and Softmax?

Answer:
**Embeddings** are vector representations that encode the information of each input token. In the model described, each token’s final representation is obtained by summing three kinds of embeddings:

* **Token embeddings** – learned vectors for the individual WordPiece tokens.  
* **Segment embeddings** – vectors that indicate which sentence (or segment) a token belongs to.  
* **Position embeddings** – vectors that encode the token’s position in the sequence.

These summed embeddings are then fed into the transformer layers. The same weight matrix is also shared between the token‑embedding layers and the pre‑softmax linear transformation, with the weights scaled by √dₘₒ𝒹ₑₗ (Source: provided context, n/a).

**Softmax** is the activation function applied after the model produces raw logits (the output of a linear layer). It converts those logits into a probability distribution over the vocabulary, allowing the model to predict the 